In [ ]:
!pip install skrebate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gc
import os
import sys

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from skrebate import ReliefF
from sklearn.model_selection import StratifiedKFold

In [ ]:
IMP_GENES_PATH = "Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/IMP_GENES_RELIEFF/USING_KFOLD"

In [ ]:
## perform feature importance using StratifiedKFold and ReliefF
def feature_importance_reliefF(df, model, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    genes = X.columns
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    k = 1
    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold; fit ONLY on train split
        relief = ReliefF(n_neighbors=100, verbose=True)  # n_neighbors can be tuned
        relief.fit(X_train, y_train)

        # Get feature scores
        feature_scores = relief.feature_importances_
        # Map scores to gene names
        gene_importance = pd.DataFrame({
            "Gene": genes,
            "Importance": feature_scores
        })
        gene_importance = gene_importance.sort_values(by='Importance', ascending=False).reset_index(drop=True)
        gene_importance.to_csv(os.path.join(IMP_GENES_PATH, f'relieff_feature_importance_{model}_{random_state}_{k}.csv'), index=False)
        k = k+1


In [ ]:

rna_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_luad.csv'))
rna_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_lusc.csv'))

rna_luad_xena['label'] = 1
rna_lusc_xena['label'] = 0
df_rna_xena = pd.concat([rna_luad_xena, rna_lusc_xena], axis=0)

cnv_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_luad.csv'))
cnv_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_lusc.csv'))

cnv_luad_xena['label'] = 1
cnv_lusc_xena['label'] = 0
df_cnv_xena = pd.concat([cnv_luad_xena, cnv_lusc_xena], axis=0)




'''
the value of seed will also be the value of file read in later cells. For example,if current seed value is 4,
then the file read number in cell 9 & 12 will also be *_cnv_4_*.csv and *_rna_4_*.csv, respectively.
Further, in cell 25, the file saving number will be sorted_overlapping_seed4.txt.
'''
for seed in range(4,5):
    
    _df_rna_xena = df_rna_xena.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv_xena.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    feature_importance_reliefF(_df_rna_xena, 'rna', seed)
    print('Done with RNA.')
    feature_importance_reliefF(_df_cnv_xena, 'cnv',seed)
    print('Done with CNV.')


---

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

In [ ]:
ROOT = Path('Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/IMP_GENES_RELIEFF/USING_KFOLD')

In [ ]:
'''
We have executed the ReliefF method for 5 folds; each time with 80% of train data and leftover 20% of test data.
One way to combine the results is to average the importance score over all the genes and then take the intersection.
As a default, we will select top 500 genes from each omics data to identify the overlapping genes.
'''
gene_names = []
importance_score = np.zeros(24776) ## number of genes in CNV
for cnv in ROOT.glob('*_cnv_4_*.csv'):
    df = pd.read_csv(cnv)
    df_sorted = df.sort_values(by = ['Gene'])
    importance_score = importance_score + np.array(df_sorted['Importance'])
gene_names = list(df_sorted['Gene'])


In [ ]:
df_cnv = pd.DataFrame()
df_cnv['Gene'] = gene_names
df_cnv['Importance'] = importance_score

In [ ]:
df_cnv_sorted = df_cnv.sort_values(by=['Importance'], ascending=False)

In [ ]:
gene_names = []
importance_score = np.zeros(20530) ## number of genes in RNASeq
for cnv in ROOT.glob('*_rna_4_*.csv'):
    df = pd.read_csv(cnv)
    df_sorted = df.sort_values(by = ['Gene'])
    importance_score = importance_score + np.array(df_sorted['Importance'])
gene_names = list(df_sorted['Gene'])

In [ ]:
df_rna = pd.DataFrame()
df_rna['Gene'] = gene_names
df_rna['Importance'] = importance_score

In [ ]:
df_rna_sorted = df_rna.sort_values(by=['Importance'], ascending=False)

In [ ]:
## save te two dataframes

df_cnv_sorted.to_csv(ROOT/'CURATED_IMPORTANCE_CNV_4.csv', index=False)
df_rna_sorted.to_csv(ROOT/'CURATED_IMPORTANCE_RNA_4.csv', index=False)

## The above code has been executed. Now, just curate the results and find out the intersecting genes.

---

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

In [ ]:
ROOT = Path('Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/IMP_GENES_RELIEFF/USING_KFOLD')

In [ ]:
df_rna = pd.read_csv(ROOT /'CURATED_IMPORTANCE_RNA_4.csv')
df_cnv = pd.read_csv(ROOT /'CURATED_IMPORTANCE_CNV_4.csv')

In [ ]:
top_500_rna = list(df_rna['Gene'].head(500))

In [ ]:
top_500_cnv = list(df_cnv['Gene'].head(500))

In [ ]:
overlapping_genes = list(set(top_500_rna) & set(top_500_cnv))

In [ ]:
len(overlapping_genes)

In [ ]:
sorted_overlapping = sorted(overlapping_genes)

In [ ]:
sorted_overlapping

In [ ]:
with open(ROOT/'sorted_overlapping_seed4.txt', 'w') as f:
    for gene in sorted_overlapping:
        f.write(f'{gene}\n')

# The above code has been executed. We have got 5 set of genes for 5 different seed values. Now intersect those genes to identify a set of genes consistent across both omics type.

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

In [ ]:
ROOT = Path('Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/IMP_GENES_RELIEFF/USING_KFOLD')

In [ ]:
five_seed_genes = pd.read_csv(ROOT/'genes_across_five_seeds.csv')

In [ ]:
five_seed_genes

In [ ]:
common_genes = list(
    set( list(five_seed_genes['seed1']) ) &
    set( list(five_seed_genes['seed2']) ) &
    set( list(five_seed_genes['seed3']) ) &
    set( list(five_seed_genes['seed4']) ) &
    set( list(five_seed_genes['seed5']) ) 
)

In [ ]:
common_genes_final = sorted(common_genes)

In [ ]:
common_genes_final

In [ ]:
len(common_genes_final)

In [ ]:
with open(ROOT/'FINAL_COMMON_GENES.txt', 'w') as f:
    for gene in common_genes_final:
        f.write(f'{gene}\n')